# Data Quality Note

- Keep one review per order, choosing the latest by answer timestamp and then creation date.
- Keep only delivered orders.
- Use unique customer ID, not customer ID, to identify customers.
- Keep only orders from the steady period, starting January 2017.

**Cleaning Summary**
Started with 99,441 orders and 95,560 remain after cleaning (96.1%)

In [1]:
%load_ext sql
%sql duckdb:///../data/olist.duckdb
%config SqlMagic.displaylimit = 20

Connecting to 'duckdb:///../data/olist.duckdb'

# Data Quality Checks

This notebook checks the raw Olist tables before cleaning. 

In [2]:
%%sql

SELECT
    (
        SELECT COUNT(*)
        FROM (
            SELECT order_id
            FROM order_reviews
            GROUP BY order_id
            HAVING COUNT(*) > 1
        ) AS dup_orders
    ) AS orders_with_duplicates_review,
    (
        SELECT COUNT(*)
        FROM (
            SELECT review_id
            FROM order_reviews
            GROUP BY review_id
            HAVING COUNT(*) > 1
        ) AS dup_reviews
    ) AS review_on_multiple_orders

Running query in 'duckdb:///../data/olist.duckdb'

orders_with_duplicates_review,review_on_multiple_orders
547,789


**Duplicate reviews**

Neither order_id nor review_id is unique in the reviews table.

Decision: the reviews must be reduced to one per order before joining them to other tables, otherwise orders will be counted more than once.

In [3]:
%%sql

SELECT
    COUNT(*) AS orders_with_mult_reviews,
    COUNT(*) FILTER (WHERE min_score <> max_score) AS scores_differ
FROM (
    SELECT
        order_id,
        MIN(review_score) AS min_score,
        MAX(review_score) AS max_score
    FROM order_reviews
    GROUP BY order_id
    HAVING COUNT(*) > 1
) AS multi_review_orders

Running query in 'duckdb:///../data/olist.duckdb'

orders_with_mult_reviews,scores_differ
547,202


**Do the duplicate reviews have different scores?**

Decision: which review is kept changes the score, so a clear rule is needed for choosing one.

In [4]:
%%sql

SELECT order_id, review_id, review_score, review_answer_timestamp
FROM order_reviews
WHERE order_id IN (
    SELECT order_id FROM order_reviews
    GROUP BY order_id HAVING COUNT(*) > 1
)
ORDER BY order_id, review_answer_timestamp DESC
LIMIT 10

Running query in 'duckdb:///../data/olist.duckdb'

order_id,review_id,review_score,review_answer_timestamp
0035246a40f520710769010f752e7507,89a02c45c340aeeb1354a24e7d4b2c1e,5,2017-08-30 01:59:12
0035246a40f520710769010f752e7507,2a74b0559eb58fc1ff842ecc999594cb,5,2017-08-29 21:45:57
013056cfe49763c6f66bda03396c5ee3,73413b847f63e02bc752b364f6d05ee9,4,2018-03-05 17:02:00
013056cfe49763c6f66bda03396c5ee3,ab30810c29da5da8045216f0f62652a2,5,2018-02-23 12:12:30
0176a6846bcb3b0d3aa3116a9a768597,d8e8c42271c8fb67b9dad95d98c8ff80,5,2018-01-02 10:54:47
0176a6846bcb3b0d3aa3116a9a768597,830636803620cdf8b6ffaf1b2f6e92b2,5,2018-01-02 10:54:06
02355020fd0a40a0d56df9f6ff060413,017f0e1ea6386de662cbeba299c59ad1,1,2018-03-30 03:16:19
02355020fd0a40a0d56df9f6ff060413,0c8e7347f1cdd2aede37371543e3d163,3,2018-03-22 01:32:08
029863af4b968de1e5d6a82782e662f5,61fe4e7d1ae801bbe169eb67b86c6eda,4,2017-07-20 12:06:11
029863af4b968de1e5d6a82782e662f5,04d945e95c788a3aa1ffbee42105637b,5,2017-07-17 13:58:06


**Sample of orders with several reviews**

In [5]:
%%sql

SELECT
    COUNT(*) AS rows_after,
    COUNT(DISTINCT order_id) AS distinct_orders
FROM (
    SELECT *
    FROM order_reviews
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY order_id
        ORDER BY review_answer_timestamp DESC, review_creation_date DESC
    ) = 1
) 


Running query in 'duckdb:///../data/olist.duckdb'

rows_after,distinct_orders
98673,98673


Sorting by answer timestamp first and creation date second to break ties.

In [6]:
%%sql

SELECT 'orders and customers' AS join_check, COUNT(*) AS orphans
FROM orders AS o
LEFT JOIN customers AS c ON o.customer_id = c.customer_id
WHERE c.customer_id IS NULL

UNION ALL

SELECT 'order_items and orders', COUNT(*)
FROM order_items AS i
LEFT JOIN orders AS o ON i.order_id = o.order_id
WHERE o.order_id IS NULL

UNION ALL

SELECT 'order_items and products', COUNT(*)
FROM order_items AS i
LEFT JOIN products AS p ON i.product_id = p.product_id
WHERE p.product_id IS NULL

UNION ALL

SELECT 'order_items and sellers', COUNT(*)
FROM order_items AS i
LEFT JOIN sellers AS s ON i.seller_id = s.seller_id
WHERE s.seller_id IS NULL

UNION ALL

SELECT 'order_reviews and orders', COUNT(*)
FROM order_reviews AS r
LEFT JOIN orders AS o ON r.order_id = o.order_id
WHERE o.order_id IS NULL

UNION ALL

SELECT 'orders with no review', COUNT(*)
FROM orders AS o
LEFT JOIN order_reviews AS r ON o.order_id = r.order_id
WHERE r.order_id IS NULL

UNION ALL

SELECT 'orders with no items', COUNT(*)
FROM orders AS o
LEFT JOIN order_items AS i ON o.order_id = i.order_id
WHERE i.order_id IS NULL


Running query in 'duckdb:///../data/olist.duckdb'

join_check,orphans
orders and customers,0
order_items and orders,0
order_items and products,0
order_items and sellers,0
order_reviews and orders,0
orders with no review,768
orders with no items,775


**Do the tables link up correctly?**

Every review belongs to a real order. However, 768 orders have no review and 775 orders have no items.

Decision: the keys are reliable and joins are safe. Orders without a review or without items will drop.

In [7]:
%%sql

select count(distinct customer_id) as customer_ids,
count (distinct customer_unique_id) as customer_unique_ids
from customers

Running query in 'duckdb:///../data/olist.duckdb'

customer_ids,customer_unique_ids
99441,96096


customer ID does not identify a person

In [8]:
%%sql
select max(n)
from (
    select customer_id, count(*) as n
    from orders
    group by customer_id
)

Running query in 'duckdb:///../data/olist.duckdb'

max(n)
1


customer ID works as an order level key, not as a customer.

In [9]:
%%sql
select count(*)
from (
    select c.customer_unique_id
    from orders as o
    join customers as c using (customer_id)
    group by c.customer_unique_id
    having count(*) > 1
)

Running query in 'duckdb:///../data/olist.duckdb'

count_star()
2997


**Real customers with more than one order**

Finding: 2,997 real customers, identified by unique customer ID, placed two or more orders.

In [10]:
%%sql

SELECT
    (
        SELECT MAX(n)
        FROM (
            SELECT customer_id, COUNT(*) AS n
            FROM orders
            GROUP BY customer_id
        ) AS order_per_customer_id
    ) AS max_orders_per_customer_id,
    (
        SELECT COUNT(*)
        FROM (
            SELECT c.customer_unique_id
            FROM orders AS o
            JOIN customers AS c USING (customer_id)
            GROUP BY c.customer_unique_id
            HAVING COUNT(*) > 1
        ) AS repeat_customers
    ) AS people_with_repeat_orders

Running query in 'duckdb:///../data/olist.duckdb'

max_orders_per_customer_id,people_with_repeat_orders
1,2997


In [11]:
%%sql

SELECT
    order_status,
    COUNT(*) AS orders,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
FROM orders
GROUP BY order_status
ORDER BY orders DESC

Running query in 'duckdb:///../data/olist.duckdb'

order_status,orders,pct
delivered,96478,97.0
shipped,1107,1.1
canceled,625,0.6
unavailable,609,0.6
invoiced,314,0.3
processing,301,0.3
created,5,0.0
approved,2,0.0


**Order status breakdown**

In [12]:
%%sql

SELECT
    DATE_TRUNC('month', order_purchase_timestamp) AS month,
    COUNT(*) AS orders
FROM orders
GROUP BY month
ORDER BY month

Running query in 'duckdb:///../data/olist.duckdb'

month,orders
2016-09-01 00:00:00,4
2016-10-01 00:00:00,324
2016-12-01 00:00:00,1
2017-01-01 00:00:00,800
2017-02-01 00:00:00,1780
2017-03-01 00:00:00,2682
2017-04-01 00:00:00,2404
2017-05-01 00:00:00,3700
2017-06-01 00:00:00,3245
2017-07-01 00:00:00,4026


start the analysis from January 2017. 

In [13]:
%sql --close duckdb:///../data/olist.duckdb